In [20]:
import pandas as pd

df = pd.read_csv('./text/amazon_cells_labelled.txt', sep='\t', header=None, names=["review", "sentiment"])
df.head()

,review,sentiment
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


In [21]:
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re

stop_words = set(stopwords.words('english'))

def clean(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if re.match("^[a-z0-9]+$", t)]
    cleaned_tokens = [t for t in tokens if t not in string.punctuation and t not in stop_words]
    return cleaned_tokens

df['tokens'] = df['review'].apply(clean)

In [22]:
vocab = sorted(set([word for tokens in df['tokens'] for word in tokens]))

In [47]:
tf_list = []
for tokens in df['tokens']:
    total_terms = len(tokens)
    if total_terms != 0:
        
        tf = {word: tokens.count(word)/total_terms for word in vocab}
        tf_list.append(tf)

tf_df = pd.DataFrame(tf_list)
display(tf_df.head().style)

In [48]:
import math

N = len(df)
idf = {}
for word in vocab:
    doc_count = sum(word in tokens for tokens in df['tokens'])
    idf[word] = math.log(N / (1 + doc_count)) + 1

idf_df = pd.DataFrame([idf])
print("\nIDF Table:")
display(idf_df.style)


IDF Table:


In [49]:
tfidf_list = []
for tf in tf_list:
    tfidf = {word: tf[word] * idf[word] for word in vocab}
    tfidf_list.append(tfidf)

tfidf_df = pd.DataFrame(tfidf_list)
print("\nTF-IDF Table:")
display(tfidf_df.head().style)



TF-IDF Table:


In [50]:
top_n = 3
bottom_n = 3

for i, row in tfidf_df.head(5).iterrows():
    # Sort TF-IDF values descending
    sorted_row = row.sort_values(ascending=False)
    
    top_words = sorted_row.head(top_n)
    bottom_words = sorted_row.tail(bottom_n)
    
    print(f"Review {i+1}:")
    print("  Top words:", list(top_words.index))
    print("  Low words:", list(bottom_words.index))
    print()


Review 1:
  Top words: ['converter', 'us', 'unless']
  Low words: ['failed', 'factor', 'favorite']

Review 2:
  Top words: ['value', 'excellent', 'case']
  Low words: ['failed', 'factor', 'favorite']

Review 3:
  Top words: ['jawbone', 'great', 'premium']
  Low words: ['factor', 'fact', 'father']

Review 4:
  Top words: ['tied', '45', 'lasting']
  Low words: ['failed', 'factor', 'favorite']

Review 5:
  Top words: ['mic', 'great', 'premium']
  Low words: ['factor', 'fact', 'father']



In [52]:
for i in range(5):
    row = tfidf_df.iloc[i]
    top_terms = row.sort_values(ascending=False).head(3)

    print(f"\nDocument {i} (Sentiment: {df['sentiment'].iloc[i]}):")
    for word, value in top_terms.items():
        print(f" {word}: {value:.4f}")


Document 0 (Sentiment: 0):
 converter: 1.2024
 us: 1.1349
 unless: 1.1349

Document 1 (Sentiment: 1):
 value: 1.5290
 excellent: 1.1439
 case: 1.1266

Document 2 (Sentiment: 1):
 jawbone: 3.2607
 great: 1.6665
 premium: 0.0000

Document 3 (Sentiment: 0):
 tied: 1.2024
 45: 1.2024
 lasting: 1.1349

Document 4 (Sentiment: 1):
 mic: 3.1492
 great: 1.6665
 premium: 0.0000
